# Experiment: Exploración del corpus Doppler

**Pregunta.** ¿Cómo están distribuidos clases, duraciones, sample rates y cómo se ve una sirena frente a tráfico en el dominio tiempo-frecuencia?

**Criterio de éxito.** Tablas de balance, histogramas y al menos un ejemplo de waveform + STFT por clase disponible, guardados en `reports/figures/`.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

SEED = 7

def find_repo_root() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/kaggle/working"),
        Path("/kaggle/input/doppler-ml"),
    ]
    for parent in Path.cwd().resolve().parents:
        candidates.append(parent)
    for cand in candidates:
        if (cand / "src" / "paths.py").exists():
            return cand
    return Path.cwd()

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.paths import corpus_paths, figures_dir, is_kaggle, tables_dir

print("repo:", REPO_ROOT)
print("kaggle:", is_kaggle())
print("available:", list(corpus_paths().available()))
SEED


repo: /home/jeancdevx/dev/doppler/doppler-ml
kaggle: False
available: ['sirennet', 'lssiren', 'urbansound8k']


7

## Plan

- Hipótesis: sireNNet está casi balanceado en 4 clases; LSSiren es 50/50 binario; UrbanSound8K está desbalanceado y `siren` no distingue tipo.
- Variables: `label`, `duration_s`, `sr`, `n_channels`.
- Métricas: conteos, medianas, % mono, nº de `fsID` únicos vs slices.


In [2]:
import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from src.audio_features import wav_probe
from src.inventory import scan_lssiren, scan_sirennet, scan_urbansound8k

sns.set_theme(style="whitegrid", context="notebook")
rng = np.random.default_rng(SEED)
FIG = figures_dir()
TAB = tables_dir()
plt.rcParams["figure.dpi"] = 120

inv_path = TAB / "file_inventory.csv"
if inv_path.exists():
    inventory = pd.read_csv(inv_path)
else:
    paths = corpus_paths()
    frames = []
    if paths.sirennet:
        frames.append(scan_sirennet(paths.sirennet))
    if paths.lssiren:
        frames.append(scan_lssiren(paths.lssiren))
    if paths.urbansound8k:
        frames.append(scan_urbansound8k(paths.urbansound8k))
    inventory = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    if not inventory.empty:
        inventory.to_csv(inv_path, index=False)

print(inventory.groupby(["corpus", "label"]).size() if not inventory.empty else "inventory empty")
inventory.head()


corpus        label           
lssiren       road_noise           902
              siren                932
sirennet      ambulance            400
              firetruck            400
              police               454
              traffic              421
urbansound8k  air_conditioner     1000
              car_horn             429
              children_playing    1000
              dog_bark            1000
              drilling            1000
              engine_idling       1000
              gun_shot             374
              jackhammer          1000
              siren                929
              street_music        1000
dtype: int64


,corpus,path,relpath,label,task,source,fold,fsID,classID,salience
0,sirennet,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,police/sound_658.wav,police,multiclass,NaN,NaN,NaN,NaN,NaN
1,sirennet,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,police/sound_647.wav,police,multiclass,NaN,NaN,NaN,NaN,NaN
2,sirennet,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,police/sound_791_1.wav,police,multiclass,NaN,NaN,NaN,NaN,NaN
3,sirennet,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,police/sound_785_1.wav,police,multiclass,NaN,NaN,NaN,NaN,NaN
4,sirennet,/home/jeancdevx/dev/doppler/doppler-ml/data/ra...,police/sound_685_1.wav,police,multiclass,NaN,NaN,NaN,NaN,NaN


## Balance de clases


In [3]:
if inventory.empty:
    print("No hay inventario.")
    counts = pd.DataFrame(columns=["corpus", "label", "n"])
else:
    counts = inventory.groupby(["corpus", "label"]).size().reset_index(name="n")
    n_corpus = max(int(counts["corpus"].nunique()), 1)
    fig, axes = plt.subplots(1, n_corpus, figsize=(4.2 * n_corpus, 4), squeeze=False)
    for ax, (corpus, sub) in zip(axes[0], counts.groupby("corpus")):
        sns.barplot(data=sub.sort_values("n", ascending=False), x="label", y="n", ax=ax, color="#3b6d9a")
        ax.set_title(corpus)
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    fig.savefig(FIG / "class_balance.png", bbox_inches="tight")
    plt.show()
counts


/tmp/ipykernel_17792/2174148442.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,corpus,label,n
0,lssiren,road_noise,902
1,lssiren,siren,932
2,sirennet,ambulance,400
3,sirennet,firetruck,400
4,sirennet,police,454
5,sirennet,traffic,421
6,urbansound8k,air_conditioner,1000
7,urbansound8k,car_horn,429
8,urbansound8k,children_playing,1000
9,urbansound8k,dog_bark,1000


## Duración, sample rate y canales

Se sondean los WAV existentes. Si UrbanSound8K o LSSiren están solo como CSV, esas filas se omiten del probe.


In [4]:
from pathlib import Path as _P

audio_rows = inventory.copy()
audio_rows = audio_rows[audio_rows["path"].map(lambda p: _P(str(p)).exists() and _P(str(p)).suffix.lower() in {".wav", ".mp3", ".flac"})]

def probe_row(path: str) -> dict:
    try:
        return wav_probe(path)
    except Exception as exc:
        return {"sr": None, "n_channels": None, "n_samples": None, "duration_s": None, "dtype": str(exc)}

probes = []
for _, row in audio_rows.iterrows():
    meta = probe_row(row["path"])
    meta.update({"corpus": row["corpus"], "label": row["label"], "path": row["path"]})
    probes.append(meta)

probe_df = pd.DataFrame(probes)
if not probe_df.empty:
    probe_df.to_csv(TAB / "wav_probe.csv", index=False)
    summary = probe_df.groupby(["corpus", "label"]).agg(
        n=("duration_s", "count"),
        duration_median=("duration_s", "median"),
        duration_mean=("duration_s", "mean"),
        sr_median=("sr", "median"),
        channels_mean=("n_channels", "mean"),
    ).reset_index()
    summary.to_csv(TAB / "wav_probe_summary.csv", index=False)
else:
    summary = pd.DataFrame()
summary


,corpus,label,n,duration_median,duration_mean,sr_median,channels_mean
0,sirennet,ambulance,400,3.000091,3.001898,44100.0,2.0
1,sirennet,firetruck,400,3.000091,3.002029,44100.0,2.0
2,sirennet,police,454,3.000045,3.000045,44100.0,2.0
3,sirennet,traffic,421,3.000000,3.000000,44100.0,2.0


In [5]:
if not probe_df.empty:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sns.histplot(data=probe_df, x="duration_s", hue="corpus", bins=40, ax=ax, element="step")
    ax.set_xlabel("Duración (s)")
    fig.tight_layout()
    fig.savefig(FIG / "duration_hist.png", bbox_inches="tight")
    plt.show()
else:
    print("Sin WAV para histograma de duración")


/tmp/ipykernel_17792/2632699863.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## UrbanSound8K: slices vs recordings

Aunque no haya audio, el CSV oficial permite medir el riesgo de data leakage por `fsID`.


In [7]:
from src.paths import resolve_corpus

us_root = resolve_corpus("urbansound8k")
us_csv = None
if us_root:
    hits = list(us_root.rglob("UrbanSound8K.csv"))
    us_csv = hits[0] if hits else None

if us_csv is None:
    print("UrbanSound8K.csv no montado")
    us_meta = pd.DataFrame()
else:
    us_meta = pd.read_csv(us_csv)
    if "class" not in us_meta.columns and "class_name" in us_meta.columns:
        us_meta = us_meta.rename(columns={"class_name": "class"})
    leakage = us_meta.groupby("class").agg(n_slices=("slice_file_name", "nunique"), n_recordings=("fsID", "nunique")).reset_index()
    leakage["slices_per_recording"] = leakage["n_slices"] / leakage["n_recordings"]
    leakage.to_csv(TAB / "urbansound8k_slices_vs_recordings.csv", index=False)
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(data=us_meta["class"].value_counts().rename_axis("class").reset_index(name="n"), x="class", y="n", ax=ax, color="#6b4c9a")
    ax.tick_params(axis="x", rotation=45)
    ax.set_title("UrbanSound8K — clips por clase")
    fig.tight_layout()
    fig.savefig(FIG / "urbansound8k_class_counts.png", bbox_inches="tight")
    plt.show()
    display_df = leakage
    display_df


/tmp/ipykernel_17792/291139975.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Formas de onda y STFT por clase (sireNNet)

Un ejemplo aleatorio por etiqueta, misma escala temporal.


In [8]:
from src.audio_features import load_audio, stft_db

sirennet = audio_rows[audio_rows["corpus"] == "sirennet"] if not audio_rows.empty else pd.DataFrame()
if sirennet.empty:
    print("sireNNet WAV no montado: se omite waveform/STFT")
else:
    examples = []
    for label, sub in sirennet.groupby("label"):
        examples.append(sub.sample(1, random_state=SEED).iloc[0])
    n = len(examples)
    fig, axes = plt.subplots(n, 2, figsize=(10, 2.4 * n), squeeze=False)
    for i, row in enumerate(examples):
        clip = load_audio(row["path"], duration=3.0)
        t = np.arange(len(clip.y)) / clip.sr
        axes[i, 0].plot(t, clip.y, color="#1f3b57", lw=0.6)
        axes[i, 0].set_ylabel(row["label"])
        axes[i, 0].set_xlim(0, t[-1] if len(t) else 1)
        freqs, times, db = stft_db(clip)
        im = axes[i, 1].pcolormesh(times, freqs, db, shading="auto", cmap="magma")
        axes[i, 1].set_ylim(0, 8000)
        if i == 0:
            axes[i, 0].set_title("Waveform")
            axes[i, 1].set_title("STFT (dB)")
    axes[-1, 0].set_xlabel("t (s)")
    axes[-1, 1].set_xlabel("t (s)")
    fig.tight_layout()
    fig.savefig(FIG / "sirennet_waveform_stft.png", bbox_inches="tight")
    plt.show()


/tmp/ipykernel_17792/7792609.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Resultados

- El balance y las duraciones condicionan el padding/windowing del modelo.
- UrbanSound8K no debe partirse al azar: varios slices salen del mismo `fsID`.
- Siguiente: MFCC, log-mel y descriptores espectrales por clase.


In [9]:
result = {
    "seed": SEED,
    "n_inventory": int(len(inventory)),
    "n_wav_probed": int(len(probe_df)) if "probe_df" in globals() else 0,
    "figures": sorted(p.name for p in FIG.glob("*.png")),
}
result


{'seed': 7,
 'n_inventory': 12241,
 'n_wav_probed': 1675,
 'figures': ['class_balance.png',
  'descriptors_by_class.png',
  'duration_hist.png',
  'logmel_vs_mfcc_examples.png',
  'mfcc_means_by_class.png',
  'sirennet_waveform_stft.png',
  'urbansound8k_class_counts.png']}